# PerturbGen end-to-end (Python API)

API — **tokenized data → train (masking, then count) → embeddings → perturb**, with `save`/`load`. Everything runs in-process (no shell/`subprocess`).

```python
import perturbgen as pg

# 1) TOKENIZE your preprocessed AnnData (CPU; no encoder needed):
#    tokenized_dir = pg.tokenize(adata_path, dataset='lps_90min_perturb',
#                                reference_time='90m_LPS', ...)

# 2) BUILD THE MODEL from the tokenized data.
#    encoder_path = the pretrained scmaskgit backbone (a MODEL input, NOT used for tokenization):
model = pg.PerturbGen(tokenized_dir, source='90m_LPS', encoder_path='...scmaskgit.ckpt')

# 3) TRAIN in two steps so you pick which masking checkpoint feeds the decoder:
masking_ckpt = model.train_masking(output_dir='model')
count_ckpt   = model.train_count(output_dir='model', masking_ckpt=masking_ckpt)

# 4) SAVE / EMBED / PERTURB (checkpoints are reused; load() restores them):
model.save('model')                                    # pg.PerturbGen.load('model')
emb  = model.get_embeddings(masking_ckpt=masking_ckpt)
pred = model.perturb(genes=['ENSG00000125538'], count_ckpt=count_ckpt)
```

## Prerequisites

- A **preprocessed AnnData** (QC'd, gene IDs harmonised — see the preprocessing notebook).
- The **tokenizer reference files** shipped with PerturbGen (`gene_median`, `token_dict`, `gene_mapping`).
- The **pretrained `scmaskgit` encoder** checkpoint (the model's frozen foundation backbone — a *model* input, **not** used by tokenization). Download from Hugging Face (README).
- A **CUDA GPU** for `train`/`get_embeddings`/`perturb`. `batch_size=64` needs a ≥40 GB GPU; lower on smaller cards.

In [1]:
import os
# ~/.bashrc points every cache (HF/torch/numba/matplotlib/wandb) at
# /lustre/.../team298/dv8/.cache — a dir dv8 can no longer write to, which crashes
# training (e.g. evaluate.load('rouge')). Redirect to a writable dv8 dir BEFORE
# importing perturbgen / torch / numba / evaluate.
_CACHE = '/lustre/scratch126/cellgen/lotfollahi/dv8/.cache'
os.environ['HF_HOME']              = _CACHE + '/huggingface'
os.environ['HF_DATASETS_CACHE']    = _CACHE + '/huggingface/datasets'
os.environ['TORCH_HOME']           = _CACHE + '/torch'
os.environ['TORCH_EXTENSIONS_DIR'] = _CACHE + '/torch_extensions'
os.environ['NUMBA_CACHE_DIR']      = _CACHE + '/numba'
os.environ['MPLCONFIGDIR']         = _CACHE + '/matplotlib'
os.environ['WANDB_DIR']            = _CACHE + '/wandb'
for _d in os.environ.values():
    if _d.startswith(_CACHE):
        os.makedirs(_d, exist_ok=True)

import sys
# make sure THIS clone's perturbgen (the new API) is importable, whatever kernel you pick
sys.path.insert(0, '/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen')

import perturbgen as pg

# repo root (wherever this clone lives) -> reference the checkpoints/dicts shipped IN the repo
_REPO = os.path.dirname(os.path.dirname(pg.__file__))

# pretrained scmaskgit encoder = the model's foundation backbone (NOT a tokenization input).
# Ships in the repo (git-tracked); same file the release used (foundation_107m / checkpoint_107m), epoch=00:
ENCODER_PATH = os.path.join(
    _REPO, 'pretraining_cohort',
    '20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt')

# tokenizer reference dicts, bundled in the repo (perturbgen/pp/) so tokenisation is self-contained:
_PP = os.path.join(_REPO, 'perturbgen', 'pp')
GENE_MEDIAN  = os.path.join(_PP, 'gene_median_dict_gftokens_gc95M.pkl')
TOKEN_DICT   = os.path.join(_PP, 'token_dict_gftokens_gc95M.pkl')
GENE_MAPPING = os.path.join(_PP, 'ensembl_mapping_dict_gc95M.pkl')
ADATA_PATH = '/nfs/team361/am74/Cytomeister/Evaluation_datasets/LPS/full_lps_new.h5ad'

# tokenized-data location, defined in this always-run setup cell so TOKENIZED_DIR /
# SOURCE survive a kernel restart WITHOUT re-running the tokenize cell (the tokenize
# cell below just builds the data at this path, and is idempotent).
DATASET_NAME = 'lps_90min_perturb'
from perturbgen.configs.paths import TOKENIZED_DIR as _TOK_ROOT
TOKENIZED_DIR = os.path.join(str(_TOK_ROOT), DATASET_NAME)
SOURCE = '90m_LPS'


## 1. Tokenize your data

Tokenize + pair a **preprocessed AnnData** (raw counts in `.X`, Ensembl IDs as the `var` index) with `pg.tokenize(...)` — a CPU step that writes the `.dataset` layout the model reads. `reference_time` is the **source** state (pairs run *source → later `time_point_order`*); use `90m_LPS` for the IL1B example.

Gene selection defaults to the **released recipe**: `hvg_flavor='seurat_v3'` (raw-count variance ranking) over the Geneformer-tokenizable genes, batched by `time_obs`. This reproduces the model's original 2,000-gene tokenisation.

In [2]:
# Tokenize a preprocessed AnnData -> writes the .dataset layout the model reads.
# Defaults reproduce the released tokenisation (seurat_v3 HVG over the GF-tokenizable
# pool; cell/mito filters off). Point ADATA_PATH (cell above) at your data.
TOKENIZED_DIR = pg.tokenize(
    ADATA_PATH,
    dataset=DATASET_NAME,
    reference_time='90m_LPS',                          # source state
    time_obs='time_after_LPS',                         # also the HVG batch key
    time_point_order=['90m_LPS', '6h_LPS', '10h_LPS'],
    var_list=['cell_pairing_index', 'time_after_LPS', 'cell_type_harmonized'],
    main_pairing_obs='cell_type_harmonized',
    gene_median_path=GENE_MEDIAN,
    token_dict_path=TOKEN_DICT,
    gene_mapping_path=GENE_MAPPING,
    n_hvg=2000,
    hvg_mode='after_tokenisation',
    hvg_flavor='seurat_v3',        # raw-count variance ranking (release recipe)
    exclude_non_gf_genes=True,     # HVG over the Geneformer-tokenizable genes
    cell_gene_filter=False,
    remove_mito_ribo_genes=False,
    nproc=4,
)


[PerturbGen] tokenized dataset already exists — skipping (overwrite=True to redo): /lustre/scratch126/cellgen/lotfollahi/dv8/T_perturb/tokenized_data/lps_90min_perturb


## 2. Build the model

`PerturbGen(...)` reads the already-tokenized data (infers the tokenizer's file layout from the directory, `source`, `n_hvg`). `encoder_path` is the pretrained backbone; architecture and retained columns (`var_list`) are set here.

In [3]:
model = pg.PerturbGen(
    TOKENIZED_DIR,
    source=SOURCE,
    n_hvg=2000,
    encoder_path=ENCODER_PATH,
    n_layers=6, d_ff=32, d_model=768,          # LPS: d_ff=32
    pred_tps=[1, 2],                           # 90m baseline -> targets 6h, 10h
    var_list=['cell_type_harmonized', 'cell_pairing_index', 'time_after_LPS'],
    cond_list=['time_after_LPS'],              # time conditioning
    mask_scheduler='cosine',                   # release ckpt hparam (API default is 'pow')
    context_mode=True,                         # ON during training (you can adjust)
    seed=42,                                   # count seed; masking overridden to 0 in train_masking (release: mask s_0, count s_42)
)

## 3a. Train the masking model (GPU required)

A checkpoint is saved **for every epoch** under `lps_model/masking/checkpoints`. Inspect the training curves (e.g. in Weights & Biases) to decide which epoch to feed the count decoder. `train_masking` returns the **metric-best** checkpoint (the default used by `train_count`).

In [4]:
masking_ckpt = model.train_masking(
    output_dir='lps_model',
    max_epochs=10,
    batch_size=64,   # >=40GB GPU; lower to ~16 on 24GB cards
    cellgen_lr=1e-4, cellgen_wd=1e-4,
    seed=0,          # release masking used seed 0 (ckpt s_0)
)
print('best masking checkpoint:', masking_ckpt)

[PerturbGen] training masking model ...


Seed set to 42


Current working directory: /lustre/scratch126/cellgen/lotfollahi/dv8
Loading and preprocessing data...
Loading 2_10h_LPS.dataset...
Loading 1_6h_LPS.dataset...
Loading 1_6h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_10h_LPS.h5ad...


...

Sanity Checking: |                                                                                            …

/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:105: Total length of `list` across ranks is zero. Please make sure this was your intention.


Training: |                                                                                                   …

/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('lr', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/perplexity', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/masking_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
Epoch 0, global step 16457: 'train/perplexity' reached 161.98137 (best 161.98137), saving model to '/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/2

[PerturbGen] 10 masking checkpoint(s) under /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints
[PerturbGen] metric-best (default for train_count): /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260901_1827_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2_s_0-epoch=09.ckpt
best masking checkpoint: /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260901_1827_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2_s_0-epoch=09.ckpt


## 3b. Train the count decoder (GPU required)

By default `train_count` uses the metric-best masking checkpoint from the previous step. To use a **different epoch** (based on the masking curves), pass it explicitly, e.g.
`masking_ckpt='lps_model/masking/checkpoints/<...>-epoch=07.ckpt'`.

In [6]:
count_ckpt = model.train_count(
    output_dir='lps_model',
    # release count was initialised from the masking checkpoint at epoch=08 (s_0);
    # for exact reproduction pass it explicitly:
    masking_ckpt='lps_model/masking/checkpoints/20260901_1827_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2_s_0-epoch=08.ckpt',
    max_epochs=4,
    count_lr=1e-3, count_wd=1e-3, count_dropout=0.1,   # count ckpt: lr_0.001 wd_0.001 (epoch=03 best)
    seed=42,         # release count used seed 42 (ckpt s_42)
)
print('count decoder checkpoint:', count_ckpt)

Seed set to 42


[PerturbGen] training count decoder from /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260901_1827_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2_s_0-epoch=08.ckpt ...
Loading and preprocessing data...
Loading 2_10h_LPS.dataset...
Loading 1_6h_LPS.dataset...
Loading 1_6h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_10h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

Sanity Checking: |                                                                                            …

/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:105: Total length of `list` across ranks is zero. Please make sure this was your intention.


Training: |                                                                                                   …

/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/mse', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 3. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/emd', ..., l

count decoder checkpoint: /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/count/checkpoints/20260901_2236_cellgen_train_count_lr_0.001_wd_0.001_batch_64_drop_0.1_zinb_tp_1-2_s_42_pos_time_pos_sin_m_cosine-epoch=03.ckpt


## 4. Save / load

Saves a manifest (tokenized dir, source, hyper-parameters, **and the masking + count checkpoint paths**). After `load()` the checkpoints are restored, so you can extract embeddings or perturb **without retraining**.

In [7]:
model.save('lps_model')
# later:  model = pg.PerturbGen.load('lps_model')

## 5. Gene / cell embeddings

Embeddings come from the **masking model**, so this uses a **masking checkpoint** (defaults to the trained one, or `model.masking_ckpt` after `load()`; pass `masking_ckpt=` to choose a specific epoch). Output feeds gene-program discovery (notebook 05).

In [8]:
emb_dir = model.get_embeddings(
    output_dir='lps_embeddings',
    masking_ckpt=masking_ckpt,   # masking checkpoint (embeddings come from the masking model)
    return_gene_embs=True,
    return_cell_embs=True,
    gene_embs_condition='time_after_LPS',   # gene embs are extracted per unique value of this column
)
print('embeddings at:', emb_dir)

Seed set to 42


[PerturbGen] extracting embeddings -> /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_embeddings ...
Current working directory: /lustre/scratch126/cellgen/lotfollahi/dv8
positional encoding: time_pos_sin
Loading and preprocessing data...
Loading 2_10h_LPS.dataset...
Loading 1_6h_LPS.dataset...
Loading 1_6h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_10h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

Restoring states from the checkpoint path at /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260901_1827_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2_s_0-epoch=09.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6]
Loaded model weights from the checkpoint at /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260901_1827_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2_s_0-epoch=09.ckpt


Testing: |                                                                                                    …

---Start saving embeddings


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


End saving embeddings---
embeddings at: /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_embeddings


## 6. In-silico perturbation

Perturbation prediction uses the trained **count-decoder checkpoint** (defaults to the trained one, or `model.count_ckpt` after `load()`; pass `count_ckpt=` to use a specific one).

- `mode`: `'knockout'` (mask) | `'overexpress'` | `'delete'` | `'pad'`.
- `where`: `'source'` (edit the source state, propagate to `pred_tps`) | `'target'`.
- Pass a **list** of genes to knock several out **together**.

> Use a fresh `output_dir` so previously-computed genes aren't skipped.

In [ ]:
pred_dir = model.perturb(
    genes=['ENSG00000125538'],   # IL1B
    count_ckpt=count_ckpt,       # trained count-decoder checkpoint
    mode='knockout', where='source',
    pred_tps=[1, 2], n_samples=3,
    temperature=1.5, iterations=19,   # release LPS config (MaskGIT sampling; code defaults 2.0/18)
    output_dir='lps_perturbation',
)
print('predicted post-perturbation h5ads at:', pred_dir)

Seed set to 42


Current working directory: /lustre/scratch126/cellgen/lotfollahi/dv8
[PerturbGen] perturbing ['ENSG00000125538'] (knockout, where=source) ...
Loading 1_6h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_10h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Restoring states from the checkpoint path at /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/count/checkpoints/20260901_2236_cellgen_train_count_lr_0.001_wd_0.001_batch_64_drop_0.1_zinb_tp_1-2_s_42_pos_time_pos_sin_m_cosine-epoch=03.ckpt


Start datamodule
Define context_tps for testing


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6]
Loaded model weights from the checkpoint at /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/count/checkpoints/20260901_2236_cellgen_train_count_lr_0.001_wd_0.001_batch_64_drop_0.1_zinb_tp_1-2_s_42_pos_time_pos_sin_m_cosine-epoch=03.ckpt


Testing: |                                                                                                    …

No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells with perturbed genes ['ENSG00000125538'] found in the batch. Skipping test step.
No cells w

## Next steps

- Predicted post-perturbation AnnDatas → **post-perturbation analysis** (DEGs, program shifts, trajectory effects).
- Embedding outputs → **gene-program discovery** (notebook 05).

> Note: for the paper's *cell/gene-embedding* analyses the `normal`-source tokenization is used; re-run step 1 with `reference_time='normal'` and a different `dataset` name for those, and keep `90m_LPS` for the perturbation.